# Day 14 — Same agent, four ways: Tool Runner, LangGraph, OpenAI Agents SDK, Claude Agent SDK
**Theory:** [day-14-frameworks.md](../theory/day-14-frameworks.md) · ⏱ ~105 min · Level: medium

Each section is wrapped in `try/except` so one framework's install problem doesn't stop the others.
Frameworks move fast. If an import fails, check that framework's quickstart and pin a version.

In [ ]:
# ⚙️ Setup: works in GitHub Codespaces, locally, in Colab and on Kaggle. Run this cell first.
import os, sys, pathlib, subprocess
IN_COLAB, IN_KAGGLE = "google.colab" in sys.modules, os.path.exists("/kaggle/working")
if IN_COLAB or IN_KAGGLE:  # clone + install the course repo, read the key from notebook secrets
    if not pathlib.Path("agentic-ai-course").exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/Rahulverma5/agentic-ai-course"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "agentic-ai-course/requirements.txt"], check=True)
    try:
        if IN_COLAB:
            from google.colab import userdata
            os.environ.setdefault("ANTHROPIC_API_KEY", userdata.get("ANTHROPIC_API_KEY"))
        else:
            from kaggle_secrets import UserSecretsClient
            os.environ.setdefault("ANTHROPIC_API_KEY", UserSecretsClient().get_secret("ANTHROPIC_API_KEY"))
    except Exception as e:
        print("Add ANTHROPIC_API_KEY to the notebook secrets:", e)
    REPO = pathlib.Path("agentic-ai-course").resolve()
else:  # Codespaces / local: find the repo root
    REPO = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path[:0] = [str(REPO), str(REPO / "projects" / "shopease-support-agent")]
os.chdir(REPO)  # every path in this notebook is relative to the repo root
from course_utils import check_setup, client, MODEL, CHEAP_MODEL, ask, tracker, text_of
from shopease.data import setup_data
setup_data(quiet=True)  # creates the ShopEase sample data if it's missing
check_setup();

In [ ]:
# Install framework packages (only needed today; ~1–3 min in Codespaces)
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-frameworks.txt"], check=False)

In [ ]:
import json, time, asyncio, os, traceback
import pandas as pd
from shopease.tools import Session, get_order as _get_order, check_refund_eligibility as _check
session = Session("C001")
QUESTION = "Can I get a refund for my coffee grinder O2002?"
results = []
def record(name, fn):
    t0 = time.time()
    try:
        answer = fn()
    except Exception as e:
        answer = f"FAILED: {type(e).__name__}: {e}"
        traceback.print_exc(limit=1)
    results.append({"framework": name, "seconds": round(time.time() - t0, 1), "answer": str(answer)[:200]})
    print(f"[{name}] {answer}\n")

## 0. Baseline: your own loop (course_utils.run_agent)

In [ ]:
from course_utils import run_agent
from shopease.tools import TOOLS
two_tools = [t for t in TOOLS if t["name"] in ("get_order", "check_refund_eligibility")]
fns = {"get_order": lambda order_id: _get_order(session, order_id),
       "check_refund_eligibility": lambda order_id: _check(session, order_id)}
record("raw SDK loop", lambda: run_agent([{"role": "user", "content": QUESTION}], two_tools, fns, effort="low", verbose=False).text)

## 1. Anthropic Tool Runner: the SDK runs the loop

In [ ]:
def tool_runner():
    from anthropic import beta_tool

    @beta_tool
    def get_order(order_id: str) -> str:
        """Look up one of the customer's ShopEase orders by id, e.g. 'O2001'."""
        return _get_order(session, order_id)

    @beta_tool
    def check_refund_eligibility(order_id: str) -> str:
        """Check if an order can be refunded under ShopEase policy."""
        return _check(session, order_id)

    runner = client.beta.messages.tool_runner(model=MODEL, max_tokens=4000, tools=[get_order, check_refund_eligibility],
                                              messages=[{"role": "user", "content": QUESTION}])
    last = None
    for message in runner:
        last = message
    return text_of(last)
record("Anthropic Tool Runner", tool_runner)

## 2. LangGraph: the loop as an explicit graph

In [ ]:
def langgraph_agent():
    from langchain_anthropic import ChatAnthropic
    from langchain_core.tools import tool
    from langgraph.graph import StateGraph, MessagesState, START
    from langgraph.prebuilt import ToolNode, tools_condition

    @tool
    def get_order(order_id: str) -> str:
        """Look up one of the customer's ShopEase orders by id."""
        return _get_order(session, order_id)

    @tool
    def check_refund_eligibility(order_id: str) -> str:
        """Check if an order can be refunded under ShopEase policy."""
        return _check(session, order_id)

    tools = [get_order, check_refund_eligibility]
    llm = ChatAnthropic(model=MODEL, max_tokens=4000).bind_tools(tools)
    g = StateGraph(MessagesState)
    g.add_node("agent", lambda state: {"messages": [llm.invoke(state["messages"])]})
    g.add_node("tools", ToolNode(tools))
    g.add_edge(START, "agent")
    g.add_conditional_edges("agent", tools_condition)   # → "tools" or END
    g.add_edge("tools", "agent")
    app = g.compile()
    out = app.invoke({"messages": [("user", QUESTION)]})
    return out["messages"][-1].content
record("LangGraph", langgraph_agent)

## 3. OpenAI Agents SDK (with Claude via its LiteLLM extension)

In [ ]:
async def openai_agents_sdk():
    from agents import Agent, Runner, function_tool, set_tracing_disabled
    from agents.extensions.models.litellm_model import LitellmModel
    set_tracing_disabled(True)  # its default tracing exports to OpenAI; we have no OpenAI key here

    @function_tool
    def get_order(order_id: str) -> str:
        """Look up one of the customer's ShopEase orders by id."""
        return _get_order(session, order_id)

    @function_tool
    def check_refund_eligibility(order_id: str) -> str:
        """Check if an order can be refunded under ShopEase policy."""
        return _check(session, order_id)

    agent = Agent(name="ShopEase support", instructions="Help the customer with their orders. Check eligibility before refunds.",
                  tools=[get_order, check_refund_eligibility],
                  model=LitellmModel(model=f"anthropic/{CHEAP_MODEL}", api_key=os.environ.get("ANTHROPIC_API_KEY")))
    result = await Runner.run(agent, QUESTION)
    return result.final_output

t0 = time.time()
try:
    ans = await asyncio.wait_for(openai_agents_sdk(), timeout=180)
except Exception as e:
    ans = f"FAILED: {type(e).__name__}: {e}"
results.append({"framework": "OpenAI Agents SDK (Haiku via LiteLLM)", "seconds": round(time.time() - t0, 1), "answer": str(ans)[:200]})
print(ans)

## 4. Claude Agent SDK: the Claude Code harness as a library
Custom tools are exposed as an in-process MCP server. It needs the Claude Code CLI; if it's missing, install
it with `npm install -g @anthropic-ai/claude-code` (Node is preinstalled in the Codespace).

In [ ]:
async def claude_agent_sdk():
    from claude_agent_sdk import query, ClaudeAgentOptions, tool, create_sdk_mcp_server

    @tool("get_order", "Look up one of the customer's ShopEase orders by id", {"order_id": str})
    async def get_order(args):
        return {"content": [{"type": "text", "text": _get_order(session, args["order_id"])}]}

    @tool("check_refund_eligibility", "Check if an order can be refunded under ShopEase policy", {"order_id": str})
    async def check_refund_eligibility(args):
        return {"content": [{"type": "text", "text": _check(session, args["order_id"])}]}

    server = create_sdk_mcp_server(name="shopease", version="1.0.0", tools=[get_order, check_refund_eligibility])
    options = ClaudeAgentOptions(system_prompt="You are ShopEase support. Check eligibility before refunds.",
                                 mcp_servers={"shopease": server},
                                 allowed_tools=["mcp__shopease__get_order", "mcp__shopease__check_refund_eligibility"])
    final = ""
    async for msg in query(prompt=QUESTION, options=options):
        if hasattr(msg, "result"):
            final = msg.result
    return final

t0 = time.time()
try:
    ans = await asyncio.wait_for(claude_agent_sdk(), timeout=180)  # don't hang if the CLI can't start
except Exception as e:
    ans = f"FAILED: {type(e).__name__}: {e}"
results.append({"framework": "Claude Agent SDK", "seconds": round(time.time() - t0, 1), "answer": str(ans)[:200]})
print(ans)

## 5. Compare

In [ ]:
pd.DataFrame(results)

**Write a 1-page framework decision record (ADR):** context, options, decision, consequences. Use the table
above plus: lines of code, how easy it was to see what was sent to the model, persistence, human-in-the-loop, tracing.

## ✏️ Exercises
1. Add `interrupt_before=["tools"]` to the LangGraph compile step to get human approval before tools run.
2. Use a handoff in the OpenAI Agents SDK: triage agent → refunds agent.
3. Run the same 5 eval cases through two frameworks and compare pass rates.

## 🧾 What did this notebook cost?
Every call above was recorded by `tracker`. Make a habit of checking cost after every experiment.

In [ ]:
print(tracker.summary())
tracker.to_dataframe().tail(10) if tracker.calls else None